# Enrichment Pipeline Testing

**Purpose**: End-to-end validation of the LLM enrichment agent chain against ingested batch data.

## What this notebook does
1. Initialises the environment and verifies API keys / Ollama availability
2. Discovers all batch files under `data/batches/` and **asserts** every `.jsonl` has a matching `.meta.json` sidecar
3. Loads `EventSchema` objects from the selected batch(es)
4. Inspects pre-enrichment field completeness
5. Loads `agents.yaml` config and instantiates `BatchEnrichmentRunner`
6. Runs the full 5-agent chain: `FeatureAlignment → TaxonomyClassifier → EmotionMapper → DataQuality → Deduplication`
7. Inspects post-enrichment results and confidence scores
8. **Appends enrichment metrics** (latency, per-agent timing, token usage, `llm_enriched`, confidence) to each batch's `.meta.json` sidecar for later dashboard use
9. Produces a summary DataFrame

In [1]:
import sys
import os
import logging
from pathlib import Path

# ── Path setup ────────────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd().parent
API_ROOT  = REPO_ROOT / "services" / "api"
if str(API_ROOT) not in sys.path:
    sys.path.insert(0, str(API_ROOT))

# ── Load .env ─────────────────────────────────────────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv(API_ROOT / ".env")
    print("dotenv loaded")
except ImportError:
    print("python-dotenv not installed — skipping .env load")

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)
logger = logging.getLogger("notebook.enrichment")

# ── Paths ─────────────────────────────────────────────────────────────────────
BATCH_DIR = REPO_ROOT / "data" / "batches"
print(f"API_ROOT  : {API_ROOT}")
print(f"BATCH_DIR : {BATCH_DIR}")

# ── Provider checks ───────────────────────────────────────────────────────────
import urllib.request

openai_key = os.environ.get("OPENAI_API_KEY", "")
anthropic_key = os.environ.get("ANTHROPIC_API_KEY", "")

print(f"OPENAI_API_KEY    : {'set' if openai_key else 'NOT SET'}")
print(f"ANTHROPIC_API_KEY : {'set' if anthropic_key else 'NOT SET'}")

try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as resp:
        ollama_ok = resp.status == 200
except Exception:
    ollama_ok = False
print(f"Ollama (localhost:11434) : {'reachable ✓' if ollama_ok else 'NOT reachable — start with: ollama serve'}")

dotenv loaded
API_ROOT  : /Users/josegarcia/Documents/GitHub/event-intelligence-platform/services/api
BATCH_DIR : /Users/josegarcia/Documents/GitHub/event-intelligence-platform/data/batches
OPENAI_API_KEY    : set
ANTHROPIC_API_KEY : NOT SET
Ollama (localhost:11434) : reachable ✓


## Batch Discovery & Sidecar Assertion

Every `.jsonl` batch file **must** have a corresponding `.meta.json` sidecar.  
This cell raises `AssertionError` listing all orphaned files if any are missing.

In [2]:
from src.ingestion.batch_writer import BatchWriter

writer = BatchWriter(batch_dir=BATCH_DIR)

# ── Discover all batches ───────────────────────────────────────────────────────
all_jsonl = writer.list_batches()  # all sources, all dates
print(f"Found {len(all_jsonl)} batch file(s) across all sources")
print()

# Group by source for display
by_source: dict[str, list[Path]] = {}
for p in all_jsonl:
    src = p.parent.name
    by_source.setdefault(src, []).append(p)

for src, paths in sorted(by_source.items()):
    print(f"  {src}: {len(paths)} batch(es)")
    for p in paths:
        print(f"    {p.name}")
print()

# ── Assert: every .jsonl has a .meta.json ─────────────────────────────────────
orphaned = []
for jsonl_path in all_jsonl:
    meta_path = jsonl_path.with_suffix("").with_suffix(".meta.json")
    if not meta_path.exists():
        orphaned.append(jsonl_path)

if orphaned:
    msg = "Missing .meta.json sidecar(s):\n" + "\n".join(f"  {p}" for p in orphaned)
    raise AssertionError(msg)

print(f"✅ All {len(all_jsonl)} batch(es) have a .meta.json sidecar — assertion passed.")

Found 1 batch file(s) across all sources

  ticketmaster: 1 batch(es)
    2026-02-28_ticketmaster_20260228_200150_39493678.jsonl

✅ All 1 batch(es) have a .meta.json sidecar — assertion passed.


## Select Batches & Load Events

By default, all available batches are selected.  
Set `MAX_EVENTS_PER_BATCH` to limit events for a quick smoke test (e.g. `20`), or `None` to use all.

In [3]:
import json
from src.schemas.event import EventSchema

# ── Config ────────────────────────────────────────────────────────────────────
# Set to an integer (e.g. 20) for a quick smoke test, or None to enrich all events
MAX_EVENTS_PER_BATCH: int | None = 20

# ── Load ──────────────────────────────────────────────────────────────────────
# batches_to_run: list of (source_name, jsonl_path) tuples for selected batches.
# All available batches are selected by default; slice all_jsonl here to narrow.
batches_to_run: list[tuple[str, Path]] = [
    (p.parent.name, p) for p in all_jsonl
]

# Map jsonl_path → list[EventSchema] so we can run per-batch enrichment
batch_events: dict[Path, list[EventSchema]] = {}
total_loaded = 0

for source_name, jsonl_path in batches_to_run:
    raw_records = writer.load_batch(jsonl_path)
    events: list[EventSchema] = []
    parse_errors = 0
    for rec in raw_records:
        try:
            events.append(EventSchema.model_validate(rec))
        except Exception as e:
            parse_errors += 1
            logger.warning(f"[{source_name}] Failed to parse event: {e}")

    if MAX_EVENTS_PER_BATCH is not None:
        events = events[:MAX_EVENTS_PER_BATCH]

    batch_events[jsonl_path] = events
    total_loaded += len(events)
    print(f"[{source_name}] {jsonl_path.name}: loaded {len(events)} events (parse_errors={parse_errors})")

print(f"\nTotal events to enrich: {total_loaded}")

[ticketmaster] 2026-02-28_ticketmaster_20260228_200150_39493678.jsonl: loaded 20 events (parse_errors=0)

Total events to enrich: 20


## Pre-Enrichment Field Completeness

Baseline check: how many events already have values for the fields the agents will populate.

In [4]:
# Agent → fields it is responsible for writing (mirrors agents.yaml target_fields)
AGENT_FIELDS: dict[str, list[str]] = {
    "feature_alignment": [
        "event_type", "tags", "format",
    ],
    "taxonomy_classifier": [
        "primary_category", "subcategory",
        "activity_id", "activity_name",
        "energy_level", "social_intensity", "cognitive_load",
        "physical_involvement", "repeatability",
        "unconstrained_primary_category", "unconstrained_subcategory", "unconstrained_activity",
    ],
    "emotion_mapper": [
        "emotional_output", "cost_level", "environment",
        "risk_level", "age_accessibility", "time_scale",
    ],
    "data_quality": [
        "data_quality_score",
    ],
}


def _field_filled_count(events: list, field: str) -> int:
    """Count how many events have a non-empty value for `field` (checks taxonomy_dimension too)."""
    n = 0
    for ev in events:
        val = getattr(ev, field, None)
        if val is None and ev.taxonomy_dimension:
            val = getattr(ev.taxonomy_dimension, field, None)
        if val not in (None, "", [], {}):
            n += 1
    return n


all_pre_events: list[EventSchema] = [
    ev for evts in batch_events.values() for ev in evts
]
n = len(all_pre_events)

print(f"Pre-enrichment field completeness  ({n} total events)")

for agent_name, fields in AGENT_FIELDS.items():
    agent_filled_slots = sum(_field_filled_count(all_pre_events, f) for f in fields)
    agent_total_slots  = len(fields) * n
    agent_pct = agent_filled_slots / agent_total_slots * 100 if agent_total_slots else 0

    print(f"\n  [{agent_name}]  {agent_pct:.1f}% complete  "
          f"({agent_filled_slots}/{agent_total_slots} field-slots across {n} events)")
    print("  " + "─" * 55)

    for field in fields:
        filled = _field_filled_count(all_pre_events, field)
        pct    = filled / n * 100 if n else 0
        bar    = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        print(f"    {field:<32} {bar} {pct:5.1f}%  ({filled}/{n})")

Pre-enrichment field completeness  (20 total events)

  [feature_alignment]  66.7% complete  (40/60 field-slots across 20 events)
  ───────────────────────────────────────────────────────
    event_type                       ████████████████████ 100.0%  (20/20)
    tags                             ░░░░░░░░░░░░░░░░░░░░   0.0%  (0/20)
    format                           ████████████████████ 100.0%  (20/20)

  [taxonomy_classifier]  16.7% complete  (40/240 field-slots across 20 events)
  ───────────────────────────────────────────────────────
    primary_category                 ████████████████████ 100.0%  (20/20)
    subcategory                      ████████████████████ 100.0%  (20/20)
    activity_id                      ░░░░░░░░░░░░░░░░░░░░   0.0%  (0/20)
    activity_name                    ░░░░░░░░░░░░░░░░░░░░   0.0%  (0/20)
    energy_level                     ░░░░░░░░░░░░░░░░░░░░   0.0%  (0/20)
    social_intensity                 ░░░░░░░░░░░░░░░░░░░░   0.0%  (0/20)
    cognitive

## Run Enrichment Agent Chain

Each batch is processed independently so we can track per-batch metrics.  
The full chain runs: **FeatureAlignment → TaxonomyClassifier → EmotionMapper → DataQuality → Deduplication**

In [5]:
import time
from datetime import UTC, datetime

from src.agents.orchestration.pipeline_triggers import load_agents_config
from src.agents.orchestration.agent_runner import BatchEnrichmentRunner, EnrichmentRunResult
from src.agents.validation.confidence import compute_confidence_score

# ── Load config ───────────────────────────────────────────────────────────────
agents_config = load_agents_config()
global_cfg    = agents_config.get("global", {})
print(f"MCP mode  : {global_cfg.get('mcp_mode', 'default')}")
print(f"Provider  : {global_cfg.get('default_provider', 'ollama')}")
print(f"Model     : {global_cfg.get('default_model', 'llama3.2:3b')}")
print()

# ── Build runner once — reused across all batches ─────────────────────────────
# BatchEnrichmentRunner initialises all LLM clients once here; creating it
# inside the loop would rebuild the full agent chain for every batch.
runner = BatchEnrichmentRunner(agents_config=agents_config)

# ── Per-batch enrichment ──────────────────────────────────────────────────────
# enrichment_results maps jsonl_path → EnrichmentRunResult
enrichment_results: dict[Path, EnrichmentRunResult] = {}

for source_name, jsonl_path in batches_to_run:
    events = batch_events[jsonl_path]
    if not events:
        print(f"[{source_name}] {jsonl_path.name}: no events — skipping.")
        continue

    print(f"[{source_name}] {jsonl_path.name}: enriching {len(events)} events …")

    t0     = time.monotonic()
    result = await runner.run(events=events, prompt_version="active")
    elapsed = time.monotonic() - t0

    enrichment_results[jsonl_path] = result

    total_tokens = result.total_token_usage.get("total", 0)
    has_errors   = result.total_errors > 0
    llm_ran      = total_tokens > 0

    if has_errors:
        status = "⚠️  errors"
    elif llm_ran:
        status = "✅"
    else:
        status = "⚠️  LLM did not run (0 tokens)"

    print(
        f"  {status}  {elapsed:.1f}s  "
        f"{len(result.events)} events  "
        f"{result.total_errors} errors  "
        f"{total_tokens} tokens"
    )
    for ar in result.agent_results:
        skip_note = f"  ← {ar.errors[0]}" if ar.errors else ""
        print(
            f"    {'✗' if ar.errors else '✓'} [{ar.agent_name:<22}] "
            f"{ar.duration_seconds:.2f}s  tokens={ar.token_usage.get('total', 0)}"
            f"{skip_note}"
        )

print("\nEnrichment complete.")

2026-03-02 01:07:48,841 [INFO] src.agents.orchestration.agent_runner: BatchEnrichmentRunner: running feature_alignment on 20 events


MCP mode  : local
Provider  : ollama
Model     : llama3.2:1b

[ticketmaster] 2026-02-28_ticketmaster_20260228_200150_39493678.jsonl: enriching 20 events …


2026-03-02 01:07:48,916 [INFO] httpx: HTTP Request: GET http://localhost:11434/api/tags "HTTP/1.1 200 OK"
2026-03-02 01:08:06,711 [INFO] httpx: HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 01:08:22,635 [INFO] httpx: HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 01:08:41,205 [INFO] httpx: HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 01:08:41,206 [INFO] src.agents.orchestration.agent_runner: BatchEnrichmentRunner: running taxonomy_classifier on 20 events
2026-03-02 01:09:13,577 [INFO] httpx: HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 01:09:39,645 [INFO] httpx: HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 01:09:54,186 [INFO] httpx: HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-02 01:10:06,877 [INFO] httpx: HTTP Request: 

  ✅  444.9s  20 events  0 errors  51495 tokens
    ✓ [feature_alignment     ] 52.29s  tokens=9507
    ✓ [taxonomy_classifier   ] 113.93s  tokens=21243
    ✓ [emotion_mapper        ] 46.41s  tokens=9826
    ✓ [data_quality          ] 75.18s  tokens=10919
    ✓ [deduplication         ] 156.96s  tokens=0

Enrichment complete.


## Post-Enrichment Inspection

Field completeness after enrichment, sample output, and confidence scores.

In [6]:
confidence_threshold = agents_config.get("global", {}).get("confidence_threshold", 0.6)

all_post_events: list[EventSchema] = [
    ev
    for result in enrichment_results.values()
    for ev in result.events
]
n = len(all_post_events)

# ── Field completeness post-enrichment, grouped by agent ─────────────────────
print(f"Post-enrichment field completeness  ({n} total events)")

# Gather per-agent result metadata for the stats line
agent_result_map: dict[str, "AgentResult | None"] = {}
for result in enrichment_results.values():
    for ar in result.agent_results:
        agent_result_map[ar.agent_name] = ar

for agent_name, fields in AGENT_FIELDS.items():
    agent_filled_slots = sum(_field_filled_count(all_post_events, f) for f in fields)
    agent_total_slots  = len(fields) * n
    agent_pct = agent_filled_slots / agent_total_slots * 100 if agent_total_slots else 0

    ar = agent_result_map.get(agent_name)
    stats_parts = []
    if ar is not None:
        stats_parts.append(f"{ar.duration_seconds:.2f}s")
        stats_parts.append(f"tokens={ar.token_usage.get('total', 0)}")
        if ar.errors:
            stats_parts.append(f"errors={len(ar.errors)}")
        else:
            stats_parts.append("errors=0")
    stats_str = "  |  " + "  ".join(stats_parts) if stats_parts else ""

    print(f"\n  [{agent_name}]  {agent_pct:.1f}% complete  "
          f"({agent_filled_slots}/{agent_total_slots} field-slots){stats_str}")
    print("  " + "─" * 55)

    for field in fields:
        filled = _field_filled_count(all_post_events, field)
        pct    = filled / n * 100 if n else 0
        bar    = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        print(f"    {field:<32} {bar} {pct:5.1f}%  ({filled}/{n})")

print()

# ── Confidence scores ─────────────────────────────────────────────────────────
scores = [compute_confidence_score(ev) for ev in all_post_events]
if scores:
    avg_score = sum(scores) / len(scores)
    below_threshold = [s for s in scores if s < confidence_threshold]
    print(f"Confidence scores (threshold={confidence_threshold})")
    print(f"  avg score        : {avg_score:.3f}")
    print(f"  min score        : {min(scores):.3f}")
    print(f"  max score        : {max(scores):.3f}")
    print(f"  below threshold  : {len(below_threshold)}/{len(scores)} ({len(below_threshold)/len(scores)*100:.1f}%)")

print()

# ── Sample events ─────────────────────────────────────────────────────────────
print(f"Sample enriched events (first 3)")
print("-" * 55)
for ev in all_post_events[:3]:
    et = (ev.event_type.value if hasattr(ev.event_type, "value") else str(ev.event_type)) if ev.event_type else "N/A"
    cat = ev.taxonomy_dimension.primary_category if ev.taxonomy_dimension else "N/A"
    emo = ev.taxonomy_dimension.emotional_output if ev.taxonomy_dimension else "N/A"
    conf = compute_confidence_score(ev)
    print(f"  {ev.title[:55]}")
    print(f"    event_type={et}  category={cat}  emotion={emo}  confidence={conf:.2f}")
    print(f"    tags={ev.tags[:5]}")
    print()

Post-enrichment field completeness  (20 total events)

  [feature_alignment]  93.3% complete  (56/60 field-slots)  |  52.29s  tokens=9507  errors=0
  ───────────────────────────────────────────────────────
    event_type                       ████████████████████ 100.0%  (20/20)
    tags                             ████████████████░░░░  80.0%  (16/20)
    format                           ████████████████████ 100.0%  (20/20)

  [taxonomy_classifier]  56.7% complete  (136/240 field-slots)  |  113.93s  tokens=21243  errors=0
  ───────────────────────────────────────────────────────
    primary_category                 ████████████████████ 100.0%  (20/20)
    subcategory                      ████████████████████ 100.0%  (20/20)
    activity_id                      ███████████░░░░░░░░░  55.0%  (11/20)
    activity_name                    ███████████░░░░░░░░░  55.0%  (11/20)
    energy_level                     █████████████░░░░░░░  65.0%  (13/20)
    social_intensity                 ███████

## Write Enrichment Metrics to `.meta.json`

Appends an `enrichment` block to each batch's sidecar file.  
These metrics feed a future infrastructure dashboard for the enrichment layer — no external observability tools (Sentry etc.) needed.

In [7]:
def build_enrichment_metrics(
    result: EnrichmentRunResult,
    events: list[EventSchema],
    confidence_threshold: float,
    run_at: str,
) -> dict:
    """
    Build the enrichment metrics block to append to a .meta.json file.

    Fields tracked (for dashboard use):
      enrichment_run_at           — ISO timestamp of this run
      llm_enriched                — True if at least one agent completed without fatal errors
      total_enrichment_duration_s — wall-clock seconds for the full chain
      events_enriched             — number of events that completed enrichment
      total_errors                — total error count across all agents
      total_token_usage           — {prompt_tokens, completion_tokens, total}
      prompt_versions_used        — {agent_name: prompt_version}
      confidence                  — {avg, min, max, below_threshold_count, below_threshold_pct}
      agents                      — per-agent timing, tokens, error count, events processed
    """
    scores = [compute_confidence_score(ev) for ev in result.events]
    below = [s for s in scores if s < confidence_threshold]

    llm_enriched = (
        len(result.agent_results) > 0
        and all(len(ar.errors) == 0 for ar in result.agent_results)
    )

    agents_metrics: dict = {}
    for ar in result.agent_results:
        agents_metrics[ar.agent_name] = {
            "duration_seconds": round(ar.duration_seconds, 3),
            "prompt_name": ar.prompt_name,
            "prompt_version": ar.prompt_version,
            "token_usage": ar.token_usage,
            "error_count": len(ar.errors),
            "errors": ar.errors[:10],  # cap at 10 to keep file compact
            "events_processed": len(ar.events),
        }

    return {
        "enrichment_run_at": run_at,
        "llm_enriched": llm_enriched,
        "total_enrichment_duration_seconds": round(result.total_duration_seconds, 3),
        "events_enriched": len(result.events),
        "total_errors": result.total_errors,
        "total_token_usage": result.total_token_usage,
        "prompt_versions_used": result.prompt_versions_used,
        "confidence": {
            "avg_score": round(sum(scores) / len(scores), 4) if scores else 0.0,
            "min_score": round(min(scores), 4) if scores else 0.0,
            "max_score": round(max(scores), 4) if scores else 0.0,
            "below_threshold_count": len(below),
            "below_threshold_pct": round(len(below) / len(scores) * 100, 2) if scores else 0.0,
            "threshold": confidence_threshold,
        },
        "agents": agents_metrics,
    }


def append_enrichment_to_meta(jsonl_path: Path, enrichment_block: dict) -> Path:
    """Read the existing .meta.json, add/overwrite the 'enrichment' key, write back."""
    meta_path = jsonl_path.with_suffix("").with_suffix(".meta.json")
    with open(meta_path, encoding="utf-8") as f:
        meta = json.load(f)
    meta["enrichment"] = enrichment_block
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    return meta_path


# ── Write metrics for each enriched batch ─────────────────────────────────────
run_at = datetime.now(UTC).isoformat()

for source_name, jsonl_path in batches_to_run:
    if jsonl_path not in enrichment_results:
        print(f"[{source_name}] {jsonl_path.name}: no enrichment result — skipping meta update.")
        continue

    result = enrichment_results[jsonl_path]
    metrics = build_enrichment_metrics(
        result=result,
        events=batch_events[jsonl_path],
        confidence_threshold=confidence_threshold,
        run_at=run_at,
    )
    meta_path = append_enrichment_to_meta(jsonl_path, metrics)

    print(
        f"[{source_name}] {jsonl_path.name}: "
        f"metrics written → {meta_path.name}"
    )
    print(
        f"  llm_enriched={metrics['llm_enriched']}  "
        f"duration={metrics['total_enrichment_duration_seconds']:.1f}s  "
        f"avg_confidence={metrics['confidence']['avg_score']:.3f}  "
        f"total_tokens={metrics['total_token_usage'].get('total', 0)}"
    )

print("\n✅ All .meta.json files updated with enrichment metrics.")

[ticketmaster] 2026-02-28_ticketmaster_20260228_200150_39493678.jsonl: metrics written → 2026-02-28_ticketmaster_20260228_200150_39493678.meta.json
  llm_enriched=True  duration=444.9s  avg_confidence=0.807  total_tokens=51495

✅ All .meta.json files updated with enrichment metrics.


## Summary DataFrame

Cross-batch summary table for quick comparison.

In [8]:
import pandas as pd

rows = []
for source_name, jsonl_path in batches_to_run:
    if jsonl_path not in enrichment_results:
        continue
    result = enrichment_results[jsonl_path]
    scores = [compute_confidence_score(ev) for ev in result.events]
    rows.append(
        {
            "source": source_name,
            "batch": jsonl_path.stem,
            "events_enriched": len(result.events),
            "total_duration_s": round(result.total_duration_seconds, 2),
            "total_errors": result.total_errors,
            "total_tokens": result.total_token_usage.get("total", 0),
            "llm_enriched": len(result.agent_results) > 0 and all(len(ar.errors) == 0 for ar in result.agent_results),
            "avg_confidence": round(sum(scores) / len(scores), 3) if scores else 0.0,
            "below_threshold": sum(1 for s in scores if s < confidence_threshold),
            **{
                f"{ar.agent_name}_duration_s": round(ar.duration_seconds, 2)
                for ar in result.agent_results
            },
        }
    )

if rows:
    df = pd.DataFrame(rows)
    print(f"DataFrame shape: {df.shape}")
    print()
    # Core columns
    core_cols = ["source", "events_enriched", "total_duration_s", "total_errors",
                 "total_tokens", "llm_enriched", "avg_confidence", "below_threshold"]
    print(df[core_cols].to_string(index=False))
    print()
    # Per-agent timing
    timing_cols = [c for c in df.columns if c.endswith("_duration_s") and c != "total_duration_s"]
    if timing_cols:
        print("Per-agent durations (seconds):")
        print(df[["source"] + timing_cols].to_string(index=False))
else:
    print("No enrichment results to display.")

DataFrame shape: (1, 14)

      source  events_enriched  total_duration_s  total_errors  total_tokens  llm_enriched  avg_confidence  below_threshold
ticketmaster               20            444.85             0         51495          True           0.807                3

Per-agent durations (seconds):
      source  feature_alignment_duration_s  taxonomy_classifier_duration_s  emotion_mapper_duration_s  data_quality_duration_s  deduplication_duration_s
ticketmaster                         52.29                          113.93                      46.41                    75.18                    156.96


## Verify Written Metrics

Re-read the `.meta.json` files to confirm the enrichment block is correctly persisted.

In [9]:
print("Verifying written .meta.json enrichment blocks")
print("-" * 60)

for source_name, jsonl_path in batches_to_run:
    meta = writer.get_batch_metadata(jsonl_path)
    if meta is None:
        print(f"[{source_name}] ERROR: .meta.json not found")
        continue

    enrichment = meta.get("enrichment")
    if not enrichment:
        print(f"[{source_name}] WARNING: 'enrichment' key missing from meta")
        continue

    print(f"[{source_name}] {jsonl_path.name}")
    print(f"  enrichment_run_at               : {enrichment.get('enrichment_run_at')}")
    print(f"  llm_enriched                    : {enrichment.get('llm_enriched')}")
    print(f"  total_enrichment_duration_seconds: {enrichment.get('total_enrichment_duration_seconds')}")
    print(f"  events_enriched                 : {enrichment.get('events_enriched')}")
    print(f"  total_errors                    : {enrichment.get('total_errors')}")
    conf = enrichment.get("confidence", {})
    print(f"  confidence.avg_score            : {conf.get('avg_score')}")
    print(f"  confidence.below_threshold_count: {conf.get('below_threshold_count')} ({conf.get('below_threshold_pct')}%)")
    for agent_name, agent_m in enrichment.get("agents", {}).items():
        print(
            f"  agent.{agent_name:<25} "
            f"{agent_m.get('duration_seconds', 0):.3f}s "
            f"tokens={agent_m.get('token_usage', {}).get('total', 0)} "
            f"errors={agent_m.get('error_count', 0)}"
        )
    print()

Verifying written .meta.json enrichment blocks
------------------------------------------------------------
[ticketmaster] 2026-02-28_ticketmaster_20260228_200150_39493678.jsonl
  enrichment_run_at               : 2026-03-02T00:15:13.713235+00:00
  llm_enriched                    : True
  total_enrichment_duration_seconds: 444.853
  events_enriched                 : 20
  total_errors                    : 0
  confidence.avg_score            : 0.807
  confidence.below_threshold_count: 3 (15.0%)
  agent.feature_alignment         52.289s tokens=9507 errors=0
  agent.taxonomy_classifier       113.935s tokens=21243 errors=0
  agent.emotion_mapper            46.406s tokens=9826 errors=0
  agent.data_quality              75.181s tokens=10919 errors=0
  agent.deduplication             156.961s tokens=0 errors=0



## Persistence Layer — Save Enriched Events to PostgreSQL

Writes enriched `EventSchema` objects to PostgreSQL using `EventDataWriter`.  
Requires a valid `DATABASE_URL` in `services/api/.env`.

In [10]:
import psycopg2
from src.configs.settings import get_settings

settings = get_settings()
db_params = settings.get_psycopg2_params()

print("Connecting to PostgreSQL:")
print(f"  host   : {db_params['host']}")
print(f"  port   : {db_params['port']}")
print(f"  dbname : {db_params['dbname']}")
print(f"  user   : {db_params['user']}")

try:
    conn = psycopg2.connect(**db_params)
    conn.autocommit = False
    print("\n✅ Connected to PostgreSQL")
except Exception as e:
    print(f"\n❌ Failed to connect: {e}")
    conn = None

Connecting to PostgreSQL:
  host   : localhost
  port   : 5433
  dbname : event_intelligence
  user   : experience

✅ Connected to PostgreSQL


## Persist Enriched Events

Calls `EventDataWriter.persist_batch` for each batch.  
Write order enforced by the writer:
1. **Pre-pass** — upsert `event_groups` (without `primary_event_id` yet)
2. **Main pass** — per-event upsert with all child tables (location, source, organizer, price, taxonomy, tags, artists, quality audit)
3. **Post-pass** — resolve `primary_event_id` FK and recount `event_count`

Each event is its own transaction, so failures are isolated.

In [11]:
from src.ingestion.persist import EventDataWriter

persist_results: dict = {}

if conn is None:
    print("❌ No database connection — skipping persistence.")
else:
    writer_db = EventDataWriter(conn)

    for source_name, jsonl_path in batches_to_run:
        if jsonl_path not in enrichment_results:
            print(f"[{source_name}] No enrichment result — skipping.")
            continue

        result = enrichment_results[jsonl_path]
        events_to_persist = result.events

        print(f"[{source_name}] {jsonl_path.name}: persisting {len(events_to_persist)} events …")

        t0 = time.monotonic()
        persisted_count = writer_db.persist_batch(events_to_persist)
        elapsed = time.monotonic() - t0

        persist_results[jsonl_path] = {
            "persisted": persisted_count,
            "total": len(events_to_persist),
            "duration_s": round(elapsed, 3),
        }

        status = "✅" if persisted_count == len(events_to_persist) else "⚠️ "
        print(
            f"  {status} {persisted_count}/{len(events_to_persist)} events persisted  "
            f"({elapsed:.2f}s)"
        )

    print("\nPersistence complete.")

[ticketmaster] 2026-02-28_ticketmaster_20260228_200150_39493678.jsonl: persisting 20 events …
  ✅ 20/20 events persisted  (0.08s)

Persistence complete.


## Verify Persistence

Queries PostgreSQL to confirm enriched events landed correctly.  
Checks total event count by source and spot-checks sample rows.

In [12]:
if conn is None:
    print("❌ No database connection — skipping verification.")
else:
    print("Verifying persisted events in PostgreSQL")
    print("-" * 60)

    for source_name, jsonl_path in batches_to_run:
        if jsonl_path not in persist_results:
            continue

        result = enrichment_results[jsonl_path]
        event_ids = [str(ev.event_id) for ev in result.events[:5]]

        with conn.cursor() as cur:
            cur.execute(
                """
                SELECT COUNT(*) FROM events e
                JOIN sources s ON e.source_id = s.source_id
                WHERE s.source_name = %s
                """,
                (source_name,),
            )
            total_in_db = cur.fetchone()[0]

            cur.execute(
                """
                SELECT e.event_id, e.title, e.event_type, e.data_quality_score, e.updated_at
                FROM events e
                WHERE e.event_id = ANY(%s::uuid[])
                ORDER BY e.updated_at DESC
                LIMIT 3
                """,
                (event_ids,),
            )
            sample_rows = cur.fetchall()

        pr = persist_results[jsonl_path]
        print(f"[{source_name}] total events in DB for source : {total_in_db}")
        print(f"  persisted this run : {pr['persisted']}/{pr['total']}  ({pr['duration_s']:.2f}s)")
        print()
        if sample_rows:
            print("  Sample rows from DB:")
            for row in sample_rows:
                print(
                    f"    event_id={str(row[0])[:8]}…  "
                    f"title={str(row[1])[:40]}  "
                    f"type={row[2]}  "
                    f"quality={row[3]}  "
                    f"updated={row[4]}"
                )
        else:
            print("  (no rows found for sample event_ids — check event_id format)")
        print()

    conn.close()
    print("✅ Verification complete. DB connection closed.")

Verifying persisted events in PostgreSQL
------------------------------------------------------------
[ticketmaster] total events in DB for source : 20
  persisted this run : 20/20  (0.08s)

  Sample rows from DB:
    event_id=0d20a01c…  title=Vida Records & Friends: Fades + Maig  type=concert  quality=0.9  updated=2026-03-02 00:15:14.234301
    event_id=0e005a1c…  title=BLACKPANDA - Vapor y Cielo Tour (Barcelo  type=concert  quality=1.0  updated=2026-03-01 16:06:00.907722
    event_id=a02de518…  title=Vida Records & Friends: Presentació Cart  type=concert  quality=0.8  updated=2026-03-01 16:06:00.880004

✅ Verification complete. DB connection closed.
